# This is the task 03


In [ ]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Step 2: Load and Preprocess the California Housing Dataset
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['MedHouseValue'] = california.target

# Define features (X) and target (y)
X = df.drop('MedHouseValue', axis=1)
y = df['MedHouseValue']

# Split dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
# Step 3: Overfitting Analysis & K-Fold Cross-Validation
# Evaluate base Random Forest using 5-Fold Cross-Validation
rf_base = RandomForestRegressor(random_state=42)
cv_scores = cross_val_score(rf_base, X_train_scaled, y_train, cv=5, scoring='r2')

print(f"Cross-Validation R2 Scores: {cv_scores}")
print(f"Mean CV R2 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Training vs Testing Performance Check for Overfitting
rf_base.fit(X_train_scaled, y_train)
train_pred = rf_base.predict(X_train_scaled)
test_pred = rf_base.predict(X_test_scaled)

print(f"Train R2 Score: {r2_score(y_train, train_pred):.4f}")
print(f"Test R2 Score:  {r2_score(y_test, test_pred):.4f}")

In [ ]:
# Step 4: Hyperparameter Tuning using GridSearchCV
# Setting up parameter grid for Random Forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

print("Running GridSearchCV (this may take a minute)...")
grid_search.fit(X_train_scaled, y_train)

print("Best Parameters Found:")
print(grid_search.best_params_)

In [ ]:
# Step 5: Evaluate Optimized Model & Final Comparison
best_rf = grid_search.best_estimator_

# Predict with tuned model
tuned_test_pred = best_rf.predict(X_test_scaled)

tuned_mse = mean_squared_error(y_test, tuned_test_pred)
tuned_rmse = np.sqrt(tuned_mse)
tuned_mae = mean_absolute_error(y_test, tuned_test_pred)
tuned_r2 = r2_score(y_test, tuned_test_pred)

print("--- Optimized Random Forest Performance ---")
print(f"MSE:  {tuned_mse:.4f}")
print(f"RMSE: {tuned_rmse:.4f}")
print(f"MAE:  {tuned_mae:.4f}")
print(f"R2 Score: {tuned_r2:.4f}")


In [ ]:
# Step 6: Save the Final Optimized Model
joblib.dump(best_rf, 'optimized_house_price_model.pkl')
joblib.dump(scaler, 'optimized_scaler.pkl')
print("Optimized model and scaler successfully saved!")